# Stress Prediction — Pseudo-Labeling (근접중복 test 샘플 활용)

**아이디어**: test 샘플 중 train과 거리가 매우 가까운(근접중복) 샘플은, 그 최근접 이웃의
타겟값을 "거의 확실한 정답"으로 볼 수 있습니다. 이런 test 샘플들을 pseudo-label을 붙여
train에 추가하고 tree 모델을 재학습하면, tree 모델도 이 근접중복 패턴을 간접적으로
학습할 수 있을 것이라는 가설입니다.

**검증 방법(leakage 방지)**: CV의 각 fold에서, val fold는 절대 건드리지 않고
**실제 test.csv(비지도)**를 pseudo-label 후보로 사용합니다. train fold로 학습한
모델이 test.csv 중 가까운 샘플을 찾아 pseudo-label을 만들고, 그걸 train fold에
추가해서 재학습한 뒤, val fold로 평가합니다. 이러면 val fold 정보가 전혀 새지 않습니다.


In [1]:

from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from scipy.spatial import cKDTree

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 800  # 탐색용
TREE_QUANTILE = 0.54
PAIR_QUANTILE = 0.48
PAIR_WEIGHT = 0.28
PSEUDO_LABEL_THRESHOLD = 0.05  # 아주 확실한(근접중복) 매칭만 pseudo-label로 사용

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_prediction(train_features, val_features, target) -> np.ndarray:
    train_no_gender = train_features.drop(columns=["gender"])
    val_no_gender = val_features.drop(columns=["gender"])
    weighted_train = add_feature_copies(train_no_gender)
    weighted_val = add_feature_copies(val_no_gender)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    tree_predictions = np.column_stack([t.predict(val_matrix) for t in model.estimators_])
    return np.quantile(tree_predictions, TREE_QUANTILE, axis=1)


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction(train_features, val_features, target) -> np.ndarray:
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    return np.quantile(np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1)


def v34_predict(train_features, val_features, target) -> np.ndarray:
    tree_pred = tree_prediction(train_features, val_features, target)
    pair_pred = pair_prediction(train_features, val_features, target)
    return np.clip(np.round((1 - PAIR_WEIGHT) * tree_pred + PAIR_WEIGHT * pair_pred, 2), 0.0, 1.0)


def nearest_neighbor_lookup(train_raw, other_raw, target, numeric_cols):
    train_numeric = train_raw[numeric_cols].fillna(train_raw[numeric_cols].median())
    other_numeric = other_raw[numeric_cols].fillna(train_raw[numeric_cols].median())
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_numeric)
    other_scaled = scaler.transform(other_numeric)
    tree = cKDTree(train_scaled)
    nearest_dist, nearest_idx = tree.query(other_scaled, k=1)
    nearest_target = target.to_numpy(dtype=float)[nearest_idx]
    return nearest_dist, nearest_target


## 5-Fold CV — pseudo-labeling 있음 vs 없음 (val fold는 절대 건드리지 않음)

In [2]:

train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")  # 실제 비지도 test -> pseudo-label 후보 풀
y_full = train_raw[TARGET].reset_index(drop=True)
raw_input = train_raw.drop(columns=[TARGET, ID_COLUMN]).reset_index(drop=True)
test_input = test_raw.drop(columns=[ID_COLUMN]).reset_index(drop=True)
numeric_cols = raw_input.select_dtypes(include=["number"]).columns.tolist()

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)
oof_baseline = np.zeros(len(y_full))
oof_pseudo = np.zeros(len(y_full))
n_pseudo_used = []

for fold, (train_idx, val_idx) in enumerate(cv.split(raw_input), start=1):
    train_raw_fold = raw_input.iloc[train_idx].reset_index(drop=True)
    val_raw_fold = raw_input.iloc[val_idx].reset_index(drop=True)
    y_train_fold = y_full.iloc[train_idx].reset_index(drop=True)
    y_val_fold = y_full.iloc[val_idx].reset_index(drop=True)

    train_feat = add_features(train_raw_fold)
    val_feat = add_features(val_raw_fold)

    # baseline: pseudo-label 없이 그대로
    oof_baseline[val_idx] = v34_predict(train_feat, val_feat, y_train_fold)

    # test.csv(비지도)에서 train_fold와 근접중복인 샘플을 찾아 pseudo-label 부여
    nearest_dist, nearest_target = nearest_neighbor_lookup(train_raw_fold, test_input, y_train_fold, numeric_cols)
    confident_mask = nearest_dist < PSEUDO_LABEL_THRESHOLD
    n_pseudo_used.append(confident_mask.sum())

    pseudo_raw = test_input[confident_mask].reset_index(drop=True)
    pseudo_labels = pd.Series(nearest_target[confident_mask], name=TARGET)

    augmented_raw = pd.concat([train_raw_fold, pseudo_raw], ignore_index=True)
    augmented_labels = pd.concat([y_train_fold, pseudo_labels], ignore_index=True)
    augmented_feat = add_features(augmented_raw)

    oof_pseudo[val_idx] = v34_predict(augmented_feat, val_feat, augmented_labels)

    print(f"Fold {fold} done (pseudo-label 개수: {confident_mask.sum()})")

mae_baseline = mean_absolute_error(y_full, oof_baseline)
mae_pseudo = mean_absolute_error(y_full, oof_pseudo)
print()
print(f"pseudo-labeling 없음 CV MAE: {mae_baseline:.6f}")
print(f"pseudo-labeling 있음 CV MAE: {mae_pseudo:.6f}")
print(f"평균 pseudo-label 사용 개수: {np.mean(n_pseudo_used):.0f}")
print(f"개선폭: {mae_baseline - mae_pseudo:+.6f}")


Fold 1 done (pseudo-label 개수: 123)
Fold 2 done (pseudo-label 개수: 112)
Fold 3 done (pseudo-label 개수: 129)
Fold 4 done (pseudo-label 개수: 115)
Fold 5 done (pseudo-label 개수: 113)

pseudo-labeling 없음 CV MAE: 0.147697
pseudo-labeling 있음 CV MAE: 0.148187
평균 pseudo-label 사용 개수: 118
개선폭: -0.000490


## 최종 제출 파일 생성 (pseudo-labeling이 확실히 도움될 때만)

In [3]:

USE_PSEUDO_LABELING = False  # 위 결과가 확실히 개선이면 True로 변경

submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
N_ESTIMATORS = 1200

if USE_PSEUDO_LABELING:
    nearest_dist, nearest_target = nearest_neighbor_lookup(raw_input, test_input, y_full, numeric_cols)
    confident_mask = nearest_dist < PSEUDO_LABEL_THRESHOLD
    pseudo_raw = test_input[confident_mask].reset_index(drop=True)
    pseudo_labels = pd.Series(nearest_target[confident_mask], name=TARGET)
    final_train_raw = pd.concat([raw_input, pseudo_raw], ignore_index=True)
    final_labels = pd.concat([y_full, pseudo_labels], ignore_index=True)
    print(f"pseudo-label로 추가된 샘플 수: {confident_mask.sum()}")
else:
    final_train_raw = raw_input
    final_labels = y_full

final_train_feat = add_features(final_train_raw)
final_test_feat = add_features(test_input)
final_prediction = v34_predict(final_train_feat, final_test_feat, final_labels)

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_pseudolabel_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Prediction range:", final_prediction.min(), final_prediction.max())
display(output.head(10))


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_pseudolabel_0806.csv
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.62
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.51
4,TEST_0004,0.53
5,TEST_0005,0.47
6,TEST_0006,0.78
7,TEST_0007,0.46
8,TEST_0008,0.57
9,TEST_0009,0.88
